# 2.12 主动学习 (Active Learning)

模型主动选择最有信息量的样本进行标注，以最少标注成本获得最大性能提升。

本节涵盖：
- 不确定性采样
- 多样性采样
- 委员会查询
- 基于期望信息增益

## 1. 不确定性采样

**目的**：选择模型最不确定的样本

**基本原理**：选择模型预测熵最高或预测概率最接近均匀分布的样本进行标注，这些样本对模型决策边界的确定最有帮助。

**不确定性度量**：
- **熵**：H(p) = -Σ p_i log(p_i)，越高越不确定
- **边缘采样**：选择最高和次高概率差最小的样本
- **最小置信度**：选择最大预测概率最低的样本

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np

torch.manual_seed(42)

class UncertaintySampler:
    def __init__(self, model):
        self.model = model

    def entropy(self, probs):
        return -(probs * torch.log(probs + 1e-10)).sum(dim=-1)

    def margin(self, probs):
        sorted_p, _ = probs.sort(dim=-1, descending=True)
        return sorted_p[:, 0] - sorted_p[:, 1]

    def least_confidence(self, probs):
        return 1 - probs.max(dim=-1)[0]

    def select(self, X, n_select=5, strategy='entropy'):
        with torch.no_grad():
            probs = F.softmax(self.model(X), dim=-1)
        if strategy == 'entropy':
            scores = self.entropy(probs)
        elif strategy == 'margin':
            scores = -self.margin(probs)
        elif strategy == 'least_confidence':
            scores = self.least_confidence(probs)
        return scores.argsort(descending=True)[:n_select]

model = nn.Sequential(nn.Linear(10, 32), nn.ReLU(), nn.Linear(32, 3))
X_train = torch.randn(50, 10)
y_train = (X_train[:, 0] + X_train[:, 1] > 0).long()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

for _ in range(20):
    loss = F.cross_entropy(model(X_train), y_train)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

X_pool = torch.randn(200, 10)
sampler = UncertaintySampler(model)

print('=== Uncertainty Sampling Strategies ===')
for strategy in ['entropy', 'margin', 'least_confidence']:
    selected = sampler.select(X_pool, n_select=5, strategy=strategy)
    with torch.no_grad():
        probs = F.softmax(model(X_pool[selected]), dim=-1)
    print(f'\n{strategy.upper()}:')
    for i, idx in enumerate(selected):
        p = probs[i].tolist()
        ent = sampler.entropy(probs[i:i+1]).item()
        print(f'  Sample {idx.item()}: probs=[{p[0]:.3f}, {p[1]:.3f}, {p[2]:.3f}], entropy={ent:.3f}')

print('\nKey: All strategies select samples where the model is most uncertain.')
print('These samples provide the most information for improving the decision boundary.')

## 2. 多样性采样

**目的**：选择最具代表性的样本

**基本原理**：在特征空间中选择最能覆盖数据分布的样本进行标注，避免标注冗余的相似样本，确保标注数据的多样性。

**方法**：
- **核心集选择**：选择能最好代表整体分布的子集
- **聚类采样**：先聚类再从每个聚类中选择代表
- **特征空间覆盖**：最大化选中样本在特征空间中的覆盖范围

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(42)

class DiversitySampler:
    def __init__(self, model):
        self.model = model

    def get_features(self, X):
        with torch.no_grad():
            x = F.relu(self.model[0](X))
            return self.model[1](x) if len(self.model) > 2 else x

    def core_set_select(self, X, n_select=10):
        features = self.get_features(X)
        features = F.normalize(features, dim=-1)
        selected = [torch.randint(len(X), (1,)).item()]
        for _ in range(n_select - 1):
            sel_features = features[selected]
            min_dist = torch.cdist(features, sel_features).min(dim=1)[0]
            next_idx = min_dist.argmax().item()
            selected.append(next_idx)
        return selected

    def cluster_select(self, X, n_clusters=5, per_cluster=2):
        features = self.get_features(X)
        n = len(X)
        centroids = features[torch.randperm(n)[:n_clusters]]
        for _ in range(10):
            dists = torch.cdist(features, centroids)
            assignments = dists.argmin(dim=1)
            for c in range(n_clusters):
                mask = assignments == c
                if mask.any():
                    centroids[c] = features[mask].mean(0)
        selected = []
        for c in range(n_clusters):
            mask = (assignments == c).nonzero().squeeze(-1)
            if len(mask) > 0:
                n_pick = min(per_cluster, len(mask))
                picked = mask[torch.randperm(len(mask))[:n_pick]]
                selected.extend(picked.tolist())
        return selected

model = nn.Sequential(nn.Linear(10, 32), nn.ReLU(), nn.Linear(32, 16), nn.ReLU(), nn.Linear(16, 3))
X_pool = torch.randn(200, 10)
sampler = DiversitySampler(model)

print('=== Diversity Sampling Strategies ===')
core_set = sampler.core_set_select(X_pool, n_select=10)
cluster_set = sampler.cluster_select(X_pool, n_clusters=5, per_cluster=2)

features = sampler.get_features(X_pool)
features = F.normalize(features, dim=-1)

def compute_coverage(selected_indices, all_features, k=5):
    sel_features = all_features[selected_indices]
    dists = torch.cdist(all_features, sel_features)
    min_dists = dists.min(dim=1)[0]
    return min_dists.mean().item()

random_indices = torch.randperm(200)[:10].tolist()
core_coverage = compute_coverage(core_set, features)
cluster_coverage = compute_coverage(cluster_set, features)
random_coverage = compute_coverage(random_indices, features)

print(f'Core-set selected: {core_set}')
print(f'Cluster selected:  {cluster_set}')
print(f'\nFeature space coverage (lower = better):')
print(f'  Core-set:  {core_coverage:.4f}')
print(f'  Cluster:   {cluster_coverage:.4f}')
print(f'  Random:    {random_coverage:.4f}')
print('\nKey: Diversity sampling ensures selected samples cover the feature space well.')

## 3. 委员会查询（Query-by-Committee）

**目的**：利用模型间分歧选择样本

**基本原理**：训练多个模型组成"委员会"，选择委员会成员预测最不一致的样本进行标注，这些样本处于决策边界附近，最有信息量。

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(42)

n_committee = 5
committee = [nn.Sequential(nn.Linear(10, 32), nn.ReLU(), nn.Linear(32, 3)) for _ in range(n_committee)]

X_train = torch.randn(50, 10)
y_train = (X_train[:, 0] + X_train[:, 1] > 0).long()

for model in committee:
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
    for _ in range(20):
        loss = F.cross_entropy(model(X_train), y_train)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

X_pool = torch.randn(200, 10)

with torch.no_grad():
    all_probs = torch.stack([F.softmax(m(X_pool), dim=-1) for m in committee])
    avg_probs = all_probs.mean(dim=0)
    disagreement = -(avg_probs * torch.log(avg_probs + 1e-10)).sum(dim=-1)
    vote_entropy = disagreement

    all_preds = torch.stack([m(X_pool).argmax(-1) for m in committee])
    n_unique = torch.tensor([len(torch.unique(all_preds[:, i])) for i in range(len(X_pool))])

top_disagree = vote_entropy.argsort(descending=True)[:5]
top_agree = vote_entropy.argsort()[:5]

print('=== Query-by-Committee ===')
print(f'Committee size: {n_committee} models\n')

print('Most disagreed samples (should be labeled):')
for idx in top_disagree:
    votes = all_preds[:, idx].tolist()
    print(f'  Sample {idx.item()}: votes={votes}, unique_votes={n_unique[idx].item()}, entropy={vote_entropy[idx]:.3f}')

print('\nMost agreed samples (less informative):')
for idx in top_agree:
    votes = all_preds[:, idx].tolist()
    print(f'  Sample {idx.item()}: votes={votes}, unique_votes={n_unique[idx].item()}, entropy={vote_entropy[idx]:.3f}')

print('\nKey: Samples with highest disagreement are most informative for labeling.')

## 4. 基于期望信息增益

**目的**：选择最能减少模型不确定性的样本

**基本原理**：估计标注每个候选样本后模型不确定性的期望减少量，选择能最大程度减少整体不确定性的样本。理论上最优但计算代价高，通常用近似方法。

**信息增益**：IG(x) = H(θ) - E_y[H(θ|x,y)]

其中H(θ)是当前模型的不确定性，E_y[H(θ|x,y)]是标注x后期望的剩余不确定性。

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(42)

class ExpectedInformationGain:
    def __init__(self, model, n_classes=3, n_mc_samples=10):
        self.model = model
        self.n_classes = n_classes
        self.n_mc_samples = n_mc_samples

    def mc_dropout_predict(self, X):
        predictions = []
        self.model.train()
        for _ in range(self.n_mc_samples):
            with torch.no_grad():
                probs = F.softmax(self.model(X), dim=-1)
                predictions.append(probs)
        self.model.eval()
        return torch.stack(predictions)

    def compute_eig(self, X):
        mc_probs = self.mc_dropout_predict(X)
        avg_probs = mc_probs.mean(dim=0)
        predictive_entropy = -(avg_probs * torch.log(avg_probs + 1e-10)).sum(dim=-1)
        avg_entropy = -(mc_probs * torch.log(mc_probs + 1e-10)).sum(dim=-1).mean(dim=0)
        eig = predictive_entropy - avg_entropy
        return eig

    def select(self, X, n_select=5):
        eig = self.compute_eig(X)
        return eig.argsort(descending=True)[:n_select], eig

class DropoutModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(10, 32), nn.Dropout(0.3), nn.ReLU(),
            nn.Linear(32, 16), nn.Dropout(0.3), nn.ReLU(),
            nn.Linear(16, 3)
        )
    def forward(self, x):
        return self.net(x)

model = DropoutModel()
X_train = torch.randn(50, 10)
y_train = (X_train[:, 0] + X_train[:, 1] > 0).long()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
for _ in range(20):
    loss = F.cross_entropy(model(X_train), y_train)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

X_pool = torch.randn(100, 10)
eig_selector = ExpectedInformationGain(model, n_mc_samples=20)
selected, eig_scores = eig_selector.select(X_pool, n_select=5)

print('=== Expected Information Gain (EIG) ===')
print(f'MC Dropout samples: {eig_selector.n_mc_samples}\n')
print('Top 5 samples by EIG:')
for idx in selected:
    mc_probs = eig_selector.mc_dropout_predict(X_pool[idx:idx+1])
    pred_var = mc_probs.var(dim=0).max().item()
    print(f'  Sample {idx.item()}: EIG={eig_scores[idx]:.4f}, max_prob_var={pred_var:.4f}')

print(f'\nEIG stats: mean={eig_scores.mean():.4f}, max={eig_scores.max():.4f}, min={eig_scores.min():.4f}')
print('\nKey: EIG measures how much a sample would reduce model uncertainty.')
print('High EIG = labeling this sample would be very informative.')